In [1]:
import os
import re
import urllib.request
from pathlib import Path
from typing import List, Optional, Tuple

import cv2
import joblib
import mediapipe as mp
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = Path("data")
MODEL_PATH = Path("cnn_pose_model.pth")
META_PATH = Path("cnn_pose_model_meta.pkl")

POSE_MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "pose_landmarker/pose_landmarker_lite/float16/latest/"
    "pose_landmarker_lite.task"
)
POSE_MODEL_PATH = Path("pose_landmarker_lite.task")

if not POSE_MODEL_PATH.exists():
    print("[INFO] Downloading pose_landmarker_lite.task ...")
    urllib.request.urlretrieve(POSE_MODEL_URL, str(POSE_MODEL_PATH))
    print("[OK] Model downloaded.")

CLASS_NAMES = ["bend", "jump", "idle"]
LABEL_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
KNOWN_ACTIONS = set(CLASS_NAMES)
ABNORMAL_LABEL = "abnormal"
ABNORMAL_THRESHOLD = 0.6

print("Device:", DEVICE)
print("Data dir:", DATA_DIR.resolve())
print("Model output:", MODEL_PATH.resolve())
print("Metadata output:", META_PATH.resolve())
print("Pose model:", POSE_MODEL_PATH.resolve())

Device: cpu
Data dir: C:\workspace\CNDPT3_N8\data
Model output: C:\workspace\CNDPT3_N8\cnn_pose_model.pth
Metadata output: C:\workspace\CNDPT3_N8\cnn_pose_model_meta.pkl
Pose model: C:\workspace\CNDPT3_N8\pose_landmarker_lite.task


In [2]:
POSE_LANDMARK_COUNT = 33
KEYPOINT_DIM = 3
EXPECTED_FEATURE_SIZE = POSE_LANDMARK_COUNT * KEYPOINT_DIM

video_pattern = re.compile(r"^(jump|bend|jumb|idle)_(\d+)\.mp4$", re.IGNORECASE)


def parse_video_info(path: Path) -> Optional[Tuple[str, int]]:
    m = video_pattern.match(path.name)
    if not m:
        return None
    cls_raw = m.group(1).lower()
    idx = int(m.group(2))
    cls = "jump" if cls_raw in {"jump", "jumb"} else cls_raw
    if cls not in LABEL_TO_ID:
        return None
    return cls, idx


def list_labeled_videos(data_dir: Path):
    items = []
    for p in sorted(data_dir.glob("*.mp4")):
        info = parse_video_info(p)
        if info is None:
            continue
        cls, idx = info
        label = LABEL_TO_ID[cls]
        items.append({"path": p, "cls": cls, "idx": idx, "label": label})
    return items


videos = list_labeled_videos(DATA_DIR)
print(f"Found {len(videos)} usable videos")
print("Sample:", [v["path"].name for v in videos[:8]])

Found 74 usable videos
Sample: ['bend_1.mp4', 'bend_10.mp4', 'bend_11.mp4', 'bend_12.mp4', 'bend_13.mp4', 'bend_14.mp4', 'bend_15.mp4', 'bend_16.mp4']


In [3]:
def create_pose_landmarker():
    """Create a PoseLandmarker using Tasks API (VIDEO mode)."""
    with open(str(POSE_MODEL_PATH), "rb") as f:
        model_data = f.read()
    base_options = mp_python.BaseOptions(model_asset_buffer=model_data)
    options = mp_vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=mp_vision.RunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    return mp_vision.PoseLandmarker.create_from_options(options)


def extract_video_feature(video_path: Path) -> Optional[np.ndarray]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"[WARN] Cannot open: {video_path.name}")
        return None

    landmarker = create_pose_landmarker()
    frame_features: List[np.ndarray] = []
    frame_idx = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        frame_idx += 1
        timestamp_ms = frame_idx * 33

        result = landmarker.detect_for_video(mp_image, timestamp_ms)

        if result.pose_landmarks and len(result.pose_landmarks) > 0:
            landmarks = result.pose_landmarks[0]
            coords = np.array([[lm.x, lm.y, lm.z] for lm in landmarks], dtype=np.float32).reshape(-1)
            if coords.size == EXPECTED_FEATURE_SIZE:
                frame_features.append(coords)

    cap.release()
    landmarker.close()

    if not frame_features:
        print(f"[WARN] No pose detected: {video_path.name}")
        return None

    arr = np.stack(frame_features, axis=0)
    mean_feat = arr.mean(axis=0)
    std_feat = arr.std(axis=0)
    min_feat = arr.min(axis=0)
    max_feat = arr.max(axis=0)

    feat = np.concatenate([mean_feat, std_feat, min_feat, max_feat], axis=0)
    mu = feat.mean()
    sigma = feat.std() + 1e-8
    feat = (feat - mu) / sigma
    return feat.astype(np.float32)

In [4]:
def split_samples_80_20(samples):
    train, test = [], []
    for cls_name in CLASS_NAMES:
        cls_samples = sorted([s for s in samples if s["cls"] == cls_name], key=lambda x: x["idx"])
        n = len(cls_samples)
        if n == 0:
            continue

        if n == 1:
            n_train = 1
        else:
            n_train = int(round(n * 0.8))
            n_train = min(max(1, n_train), n - 1)

        train.extend(cls_samples[:n_train])
        test.extend(cls_samples[n_train:])

    return train, test


all_samples = []

for i, v in enumerate(videos):
    print(f"[{i+1}/{len(videos)}] Processing {v['path'].name} ...", end=" ")
    feat = extract_video_feature(v["path"])
    if feat is None:
        continue
    all_samples.append({**v, "feature": feat})
    print("OK")

print(f"\nExtracted features: {len(all_samples)} videos")

train_samples, test_samples = split_samples_80_20(all_samples)

train_count = {cls: sum(s["cls"] == cls for s in train_samples) for cls in CLASS_NAMES}
test_count = {cls: sum(s["cls"] == cls for s in test_samples) for cls in CLASS_NAMES}

print("Train videos:", len(train_samples))
print("Test videos:", len(test_samples))
print("Train class count:", train_count)
print("Test class count:", test_count)

if len(train_samples) == 0 or len(test_samples) == 0:
    raise RuntimeError("Not enough data after feature extraction. Add videos and rerun.")

[1/74] Processing bend_1.mp4 ... OK
[2/74] Processing bend_10.mp4 ... OK
[3/74] Processing bend_11.mp4 ... OK
[4/74] Processing bend_12.mp4 ... OK
[5/74] Processing bend_13.mp4 ... OK
[6/74] Processing bend_14.mp4 ... OK
[7/74] Processing bend_15.mp4 ... OK
[8/74] Processing bend_16.mp4 ... OK
[9/74] Processing bend_17.mp4 ... OK
[10/74] Processing bend_18.mp4 ... OK
[11/74] Processing bend_19.mp4 ... OK
[12/74] Processing bend_2.mp4 ... OK
[13/74] Processing bend_20.mp4 ... OK
[14/74] Processing bend_21.mp4 ... OK
[15/74] Processing bend_22.mp4 ... OK
[16/74] Processing bend_23.mp4 ... OK
[17/74] Processing bend_24.mp4 ... OK
[18/74] Processing bend_25.mp4 ... OK
[19/74] Processing bend_26.mp4 ... OK
[20/74] Processing bend_27.mp4 ... OK
[21/74] Processing bend_28.mp4 ... OK
[22/74] Processing bend_29.mp4 ... OK
[23/74] Processing bend_3.mp4 ... OK
[24/74] Processing bend_30.mp4 ... OK
[25/74] Processing bend_31.mp4 ... OK
[26/74] Processing bend_4.mp4 ... OK
[27/74] Processing bend_5

In [5]:
X_train_np = np.stack([s["feature"] for s in train_samples], axis=0)
y_train_np = np.array([s["label"] for s in train_samples], dtype=np.int64)

X_test_np = np.stack([s["feature"] for s in test_samples], axis=0)
y_test_np = np.array([s["label"] for s in test_samples], dtype=np.int64)
test_names = [s["path"].name for s in test_samples]

# Conv1D input: (batch, channels, length) — PyTorch channel-first
X_train_t = torch.tensor(X_train_np, dtype=torch.float32).unsqueeze(1).to(DEVICE)
y_train_t = torch.tensor(y_train_np, dtype=torch.long).to(DEVICE)

X_test_t = torch.tensor(X_test_np, dtype=torch.float32).unsqueeze(1).to(DEVICE)
y_test_t = torch.tensor(y_test_np, dtype=torch.long).to(DEVICE)

print("X_train:", X_train_t.shape, "y_train:", y_train_t.shape)
print("X_test:", X_test_t.shape, "y_test:", y_test_t.shape)

X_train: torch.Size([59, 1, 396]) y_train: torch.Size([59])
X_test: torch.Size([15, 1, 396]) y_test: torch.Size([15])


In [6]:
torch.manual_seed(42)
np.random.seed(42)

num_classes = len(CLASS_NAMES)
feature_len = X_train_t.shape[2]


class PoseCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.35),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


model = PoseCNN(num_classes).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 120
PATIENCE = 10
best_loss = float("inf")
patience_counter = 0
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        preds = outputs.argmax(dim=1)
        acc = (preds == y_train_t).float().mean().item()

    if loss.item() < best_loss:
        best_loss = loss.item()
        patience_counter = 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f} | Acc: {acc:.4f}")

    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break

if best_state is not None:
    model.load_state_dict(best_state)

print("CNN model trained.")

Epoch   1 | Loss: 1.0782 | Acc: 0.4237
Epoch  10 | Loss: 0.8383 | Acc: 0.6271
Epoch  20 | Loss: 0.4856 | Acc: 0.7627
Epoch  30 | Loss: 0.1635 | Acc: 0.9661
Epoch  40 | Loss: 0.0637 | Acc: 0.9831
Epoch  50 | Loss: 0.0237 | Acc: 1.0000
Epoch  60 | Loss: 0.0104 | Acc: 1.0000
Early stopping at epoch 68
CNN model trained.


In [7]:
model.eval()
with torch.no_grad():
    logits = model(X_test_t)
    y_proba = torch.softmax(logits, dim=1).cpu().numpy()

y_pred = np.argmax(y_proba, axis=1)
y_test = y_test_np

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

present_label_ids = sorted(set(y_test.tolist()) | set(y_pred.tolist()))
present_target_names = [CLASS_NAMES[idx].capitalize() for idx in present_label_ids]

print(f"Accuracy: {acc:.4f}")
print("Confusion Matrix [rows=true, cols=pred]:")
print(cm)
print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=present_label_ids,
        target_names=present_target_names,
        zero_division=0,
    )
)

print("\nPer-video predictions (with abnormal rule):")
for name, pred, prob in zip(test_names, y_pred, y_proba):
    confidence = float(np.max(prob))
    pred_name = CLASS_NAMES[int(pred)]

    if pred_name not in KNOWN_ACTIONS or confidence < ABNORMAL_THRESHOLD:
        final_pred = ABNORMAL_LABEL
    else:
        final_pred = pred_name

    print(
        f"Video: {name} -> Predict: {final_pred} "
        f"(Base: {pred_name}, Confidence: {confidence:.2f})"
    )

Accuracy: 0.8667
Confusion Matrix [rows=true, cols=pred]:
[[6 0 0]
 [1 4 1]
 [0 0 3]]

Classification report:
              precision    recall  f1-score   support

        Bend       0.86      1.00      0.92         6
        Jump       1.00      0.67      0.80         6
        Idle       0.75      1.00      0.86         3

    accuracy                           0.87        15
   macro avg       0.87      0.89      0.86        15
weighted avg       0.89      0.87      0.86        15


Per-video predictions (with abnormal rule):
Video: bend_26.mp4 -> Predict: bend (Base: bend, Confidence: 1.00)
Video: bend_27.mp4 -> Predict: bend (Base: bend, Confidence: 1.00)
Video: bend_28.mp4 -> Predict: bend (Base: bend, Confidence: 1.00)
Video: bend_29.mp4 -> Predict: bend (Base: bend, Confidence: 1.00)
Video: bend_30.mp4 -> Predict: bend (Base: bend, Confidence: 1.00)
Video: bend_31.mp4 -> Predict: bend (Base: bend, Confidence: 1.00)
Video: jump_25.mp4 -> Predict: abnormal (Base: idle, Confidenc

In [8]:
torch.save(model.state_dict(), MODEL_PATH)

metadata = {
    "model_path": str(MODEL_PATH),
    "feature_type": "mediapipe33_xyz_mean_std_min_max_per_video",
    "feature_size": feature_len,
    "num_classes": num_classes,
    "labels": LABEL_TO_ID,
    "class_names": CLASS_NAMES,
    "abnormal_label": ABNORMAL_LABEL,
    "abnormal_threshold": ABNORMAL_THRESHOLD,
}
joblib.dump(metadata, META_PATH)

print(f"Saved model weights: {MODEL_PATH.resolve()}")
print(f"Saved metadata: {META_PATH.resolve()}")

Saved model weights: C:\workspace\CNDPT3_N8\cnn_pose_model.pth
Saved metadata: C:\workspace\CNDPT3_N8\cnn_pose_model_meta.pkl
